In [1]:
import geoai

In [2]:
train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_train.tif"
train_masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_masks.tif"
test_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_test.tif"

In [3]:
train_raster_path = geoai.download_file(train_raster_url)
train_masks_path = geoai.download_file(train_masks_url)
test_raster_path = geoai.download_file(test_raster_url)

In [4]:
geoai.print_raster_info(train_raster_path, show_preview=False)

In [5]:
geoai.view_raster(train_masks_url, nodata=0, basemap=train_raster_url)

In [6]:
geoai.view_raster(test_raster_url)

In [7]:
out_folder = "output"

In [8]:
import os

tiles_images_dir = os.path.join(out_folder, "images")  # adjust subfolder name if different

if not os.path.exists(tiles_images_dir) or len(os.listdir(tiles_images_dir)) == 0:
    tiles = geoai.export_geotiff_tiles(
        in_raster=train_raster_path,
        out_folder=out_folder,
        in_class_data=train_masks_path,
        tile_size=512,
        stride=128,
        buffer_radius=0,
    )
    print(f"Exported {len(tiles)} tiles.")
else:
    print("Tiles already exist, skipping export.")

Tiles already exist, skipping export.


In [14]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

model_output_dir = f"{out_folder}/models"
model_file = os.path.join(model_output_dir, "mask_rcnn_model.pth")

if not os.path.exists(model_file):
    geoai.train_MaskRCNN_model(
        images_dir=f"{out_folder}/images",
        labels_dir=f"{out_folder}/labels",
        output_dir=model_output_dir,
        num_channels=4,
        pretrained=True,
        batch_size=1,
        num_epochs=2,
        learning_rate=0.005,
        val_split=0.2,
    )
    print("Model training complete.")
else:
    print("Trained model already exists, skipping training.")

KeyboardInterrupt: 

In [1]:
output_path = "naip_water_prediction.geojson"
gdf = geoai.raster_to_vector(
    masks_path, output_path, min_area=1000, simplify_tolerance=1
)

NameError: name 'geoai' is not defined